# LoRA Fine-tuning — Code Review Quality Scorer

**Objectives:**
1. Understand LoRA (Low-Rank Adaptation) and why it exists
2. Generate a labeled training dataset using GPT-4o-mini
3. Fine-tune Qwen2.5-1.5B with LoRA on Apple M3 Max (MPS backend)
4. Measure the improvement: base model vs fine-tuned model

**Companion project:** [`pr_review_agent`](https://github.com/rajuclh-ai/ai-systems-engineering/tree/main/pr_review_agent) generates code review comments — this model scores them.

> **Model used:** `Qwen/Qwen2.5-1.5B-Instruct` — no license acceptance required, downloads immediately, runs fast on M3 Max. To use Llama 3.2 3B instead, accept the license at huggingface.co/meta-llama/Llama-3.2-3B-Instruct and change `MODEL_NAME` below.

## Setup

Before running:
- Copy `.env.example` → `.env` and fill in your keys
- `OPENAI_API_KEY` — for data generation (Step 2)
- `HF_TOKEN` — free at huggingface.co/settings/tokens (needed even for Qwen)

In [1]:
import gc
import json
import os
import random
import re
from collections import defaultdict
from pathlib import Path

import torch
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(dotenv_path=Path("..") / ".env")

OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]
HF_TOKEN = os.environ["HF_TOKEN"]
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"  # no license needed — swap to meta-llama/Llama-3.2-3B-Instruct if you have access

# Device detection — M3 Max uses MPS
if torch.backends.mps.is_available():
    DEVICE = "mps"
    DTYPE = torch.float16
elif torch.cuda.is_available():
    DEVICE = "cuda"
    DTYPE = torch.float16
else:
    DEVICE = "cpu"
    DTYPE = torch.float32

print(f"Device: {DEVICE} | dtype: {DTYPE}")

Device: mps | dtype: torch.float16


## Step 1 — What is LoRA?

**Full fine-tuning** updates every weight in the model. For a 3B model that's 3 billion parameters — requires ~24GB GPU RAM, hours of training.

**LoRA (Low-Rank Adaptation)** freezes the original weights and injects small trainable matrices into the attention layers:

```
Original weight matrix W  (frozen, 2048×2048 in Qwen 1.5B)
    +
LoRA: W' = W + B × A      (B is 2048×r, A is r×2048, r=16)
```

With rank `r=16`, one layer needs `2 × 2048 × 16 = 65K` trainable params instead of `2048² = 4.2M`. That's **~0.14% of parameters trained**, same task-specific result.

**Why this matters for our task:** We want to teach Qwen2.5-1.5B to output structured JSON scores for code review comments. It has no idea what `nitpick` vs `blocking` means in our schema. LoRA teaches it this mapping cheaply.

## Step 2 — Generate Training Data with GPT-4o-mini

We need labeled examples: `(comment, score, label, severity)`.

Rather than hand-labeling hundreds of examples, we use GPT-4o-mini as a synthetic data generator. This is a real-world pattern: use a capable model to generate training data for a smaller, cheaper model.

We generate across 6 categories to ensure the classifier learns a diverse set of comment types.

In [2]:
SYSTEM_PROMPT = """You are a dataset generator for training an ML classifier on code review comments.

Generate realistic, diverse code review comments with quality scores.

Return a JSON object with key "examples" containing an array. Each item must have:
- "comment": the code review comment (string)
- "score": integer 1-5
- "label": one of nitpick | suggestion | warning | blocking
- "severity": one of low | medium | high | critical

Score mapping:
  1 = nitpick,    low      — style, naming, minor formatting
  2 = suggestion, low      — improvement but not required
  3 = suggestion, medium   — noticeable issue, should fix soon
  4 = warning,    high     — bug risk, logic issue, fix before merge
  5 = blocking,   critical — security flaw, data loss, must fix
"""

CATEGORIES = [
    "security vulnerabilities (SQL injection, XSS, hardcoded secrets, auth bypass)",
    "performance issues (O(n²) nested loops, N+1 database queries, memory leaks)",
    "error handling (missing try/except, unhandled None, no input validation)",
    "logic bugs (off-by-one errors, wrong boolean conditions, silent data loss)",
    "code style (poor variable names, magic numbers, excessive nesting)",
    "documentation (missing docstrings, misleading comments, undocumented params)",
]

client = OpenAI(api_key=OPENAI_API_KEY)


def generate_batch(category: str, n: int = 20) -> list:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Generate {n} varied code review comments about: {category}. Include a mix of scores 1-5."},
        ],
        temperature=0.8,
    )
    return json.loads(response.choices[0].message.content).get("examples", [])


print("Generating training data...")
all_examples = []
for cat in CATEGORIES:
    print(f"  {cat[:55]}...")
    all_examples.extend(generate_batch(cat, n=20))

random.shuffle(all_examples)
print(f"\nTotal examples generated: {len(all_examples)}")

Generating training data...
  security vulnerabilities (SQL injection, XSS, hardcoded...
  performance issues (O(n²) nested loops, N+1 database qu...
  error handling (missing try/except, unhandled None, no ...
  logic bugs (off-by-one errors, wrong boolean conditions...
  code style (poor variable names, magic numbers, excessi...
  documentation (missing docstrings, misleading comments,...

Total examples generated: 116


## Step 3 — Explore the Dataset

In [3]:
from collections import Counter

labels = [ex["label"] for ex in all_examples]
scores = [ex["score"] for ex in all_examples]

print("Label distribution:")
for label, count in sorted(Counter(labels).items()):
    print(f"  {label:<12} {count:>4}")

print("\nScore distribution:")
for score, count in sorted(Counter(scores).items()):
    print(f"  Score {score}       {count:>4}")

print("\nSample examples:")
for ex in all_examples[:3]:
    print(f"  [{ex['label']:>10}] score={ex['score']}  '{ex['comment'][:70]}'")

Label distribution:
  blocking       18
  nitpick        10
  suggestion     59
  warning        29

Score distribution:
  Score 1         10
  Score 2         24
  Score 3         35
  Score 4         29
  Score 5         18

Sample examples:
  [suggestion] score=3  'There's no input validation for the user data. It might lead to unexpe'
  [suggestion] score=2  'Consider wrapping this in a try/except to catch any unexpected errors.'
  [  blocking] score=5  'The function `sendEmail` is missing a docstring and comments. This gap'


In [4]:
# Split 85/15 and save
split_idx = int(len(all_examples) * 0.85)
train_data = all_examples[:split_idx]
test_data = all_examples[split_idx:]

Path("../data").mkdir(exist_ok=True)

for split, data in [("train", train_data), ("test", test_data)]:
    path = f"../data/{split}.jsonl"
    with open(path, "w") as f:
        for ex in data:
            f.write(json.dumps(ex) + "\n")
    print(f"Saved {len(data)} examples → {path}")

Saved 98 examples → ../data/train.jsonl
Saved 18 examples → ../data/test.jsonl


## Step 4 — Load the Base Model

We load `Qwen2.5-1.5B-Instruct`. On M3 Max, `device_map={"":"mps"}` routes all layers to the Metal GPU. At 1.5B parameters in float16 it uses ~3GB of unified memory — well within M3 Max capacity.

Note: `dtype=` is the current API (replacing deprecated `torch_dtype=`).

In [5]:
from transformers import AutoModelForCausalLM, AutoTokenizer

print(f"Loading {MODEL_NAME}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=DTYPE,
    device_map={"": DEVICE},
    token=HF_TOKEN,
)

total_params = sum(p.numel() for p in model.parameters())
print(f"Model loaded. Total parameters: {total_params / 1e9:.2f}B")

/Users/nagarajuchittaluri/ai-developer/ml-fundamentals/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading Qwen/Qwen2.5-1.5B-Instruct...


Loading weights: 100%|██████████| 338/338 [00:00<00:00, 986.22it/s] 


Model loaded. Total parameters: 1.54B


## Step 5 — Configure and Apply LoRA

Key LoRA parameters:
- **`r` (rank):** Size of the low-rank matrices. Higher = more expressive but more params. `r=16` is the standard starting point.
- **`lora_alpha`:** Scaling factor. Typically set to `2 × r`.
- **`target_modules`:** Which layers to add adapters to. `q_proj` and `v_proj` are the query and value projections in attention — the most impactful layers to adapt.

In [6]:
from peft import LoraConfig, TaskType, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Expected: ~0.14% of parameters trainable

trainable params: 2,179,072 || all params: 1,545,893,376 || trainable%: 0.1410


## Step 6 — Format Training Examples

Qwen2.5 uses the ChatML template format (`<|im_start|>` / `<|im_end|>`). Training examples include the full conversation including the assistant answer — the model learns to predict the JSON given the system prompt + user comment.

In [7]:
from datasets import Dataset

INSTRUCTION = (
    "You are a code review classifier. Given a code review comment, "
    "output a JSON object with: score (integer 1-5), "
    "label (nitpick | suggestion | warning | blocking), "
    "and severity (low | medium | high | critical). "
    "Output only the JSON object, nothing else."
)


def format_example(ex: dict) -> str:
    """Full prompt + answer — used for training only."""
    answer = json.dumps({"score": ex["score"], "label": ex["label"], "severity": ex["severity"]})
    return (
        "<|im_start|>system\n"
        f"{INSTRUCTION}<|im_end|>\n"
        "<|im_start|>user\n"
        f"{ex['comment']}<|im_end|>\n"
        "<|im_start|>assistant\n"
        f"{answer}<|im_end|>"
    )


train_dataset = Dataset.from_list([{"text": format_example(ex)} for ex in train_data])
print(f"Training examples: {len(train_dataset)}")
print("\nFormatted example:")
print(train_dataset[0]["text"])

Training examples: 98

Formatted example:
<|im_start|>system
You are a code review classifier. Given a code review comment, output a JSON object with: score (integer 1-5), label (nitpick | suggestion | warning | blocking), and severity (low | medium | high | critical). Output only the JSON object, nothing else.<|im_end|>
<|im_start|>user
There's no input validation for the user data. It might lead to unexpected behavior. Please validate inputs.<|im_end|>
<|im_start|>assistant
{"score": 3, "label": "suggestion", "severity": "medium"}<|im_end|>


## Step 7 — Fine-tune with SFTTrainer

`SFTTrainer` (Supervised Fine-tuning Trainer) from the `trl` library wraps HuggingFace Trainer with sensible defaults for instruction tuning. It handles tokenization, packing, and the training loop.

On M3 Max, expect ~2-5 minutes for 3 epochs on ~300 examples with Qwen 1.5B.

In [8]:
from trl import SFTConfig, SFTTrainer

training_args = SFTConfig(
    output_dir="../adapter",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    learning_rate=0.0002,       # use explicit float — 2e-4 parses as string in PyYAML
    warmup_ratio=0.03,
    weight_decay=0.001,
    logging_steps=10,
    save_steps=50,
    report_to="none",
    max_length=256,             # trl 0.29+ uses max_length (not max_seq_length)
    dataset_text_field="text",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    processing_class=tokenizer,
)

print("Starting LoRA fine-tuning...")
trainer.train()

trainer.save_model("../adapter")
tokenizer.save_pretrained("../adapter")
print("Adapter saved → ../adapter")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
Tokenizing train dataset: 100%|██████████| 98/98 [00:00<00:00, 4033.74 examples/s]
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.
/Users/nagarajuchittaluri/ai-developer/ml-fundamentals/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Starting LoRA fine-tuning...


Step,Training Loss
10,2.563257
20,1.731060
30,1.065765
40,0.594778
50,0.482393
60,0.426802
70,0.406455


/Users/nagarajuchittaluri/ai-developer/ml-fundamentals/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Adapter saved → ../adapter


## Step 8 — Load the Fine-tuned Model

LoRA saves only the adapter weights (a few MB), not the full model. At inference time, we load the base model and merge the adapter on top. This is one of LoRA's key advantages — the base model is shared, adapters are swappable.

In [9]:
from peft import PeftModel

# Free training model from memory first
del model
del trainer
gc.collect()
if DEVICE == "mps":
    torch.mps.empty_cache()

print("Loading base model + LoRA adapter...")

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, dtype=DTYPE, device_map={"": DEVICE}, token=HF_TOKEN
)
finetuned_model = PeftModel.from_pretrained(base_model, "../adapter")
finetuned_model.eval()
print("Fine-tuned model ready.")

Loading base model + LoRA adapter...


Loading weights: 100%|██████████| 338/338 [00:00<00:00, 1213.40it/s]


Fine-tuned model ready.


## Step 9 — Evaluate: Base Model vs Fine-tuned

We run both models on the held-out test set and compare label accuracy and per-class F1. The base model has no knowledge of our scoring schema, so it will struggle to output the right JSON labels consistently.

In [10]:
def format_prompt(comment: str) -> str:
    """Inference prompt — stops before assistant answer."""
    return (
        "<|im_start|>system\n"
        f"{INSTRUCTION}<|im_end|>\n"
        "<|im_start|>user\n"
        f"{comment}<|im_end|>\n"
        "<|im_start|>assistant\n"
    )


def predict(m, comment: str) -> dict | None:
    inputs = tokenizer(format_prompt(comment), return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out = m.generate(
            **inputs, max_new_tokens=64, do_sample=False,
            pad_token_id=tokenizer.eos_token_id, eos_token_id=tokenizer.eos_token_id,
        )
    text = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
    match = re.search(r"\{[^}]+\}", text)
    if match:
        try:
            return json.loads(match.group())
        except json.JSONDecodeError:
            return None
    return None


def evaluate_model(m, examples, model_label):
    correct = 0
    by_label = defaultdict(lambda: {"tp": 0, "fp": 0, "fn": 0})

    for ex in examples:
        pred = predict(m, ex["comment"])
        pred_label = pred.get("label") if pred else None
        true_label = ex["label"]
        if pred_label == true_label:
            correct += 1
            by_label[true_label]["tp"] += 1
        else:
            by_label[true_label]["fn"] += 1
            if pred_label:
                by_label[pred_label]["fp"] += 1

    acc = correct / len(examples) * 100
    print(f"\n{'='*50}\n  {model_label}\n  Label Accuracy: {acc:.1f}%\n{'='*50}")
    print(f"  {'Label':<12} {'Prec':>6} {'Rec':>6} {'F1':>6}")
    print(f"  {'-'*32}")
    for lbl, c in sorted(by_label.items()):
        p = c["tp"] / (c["tp"] + c["fp"]) if (c["tp"] + c["fp"]) > 0 else 0
        r = c["tp"] / (c["tp"] + c["fn"]) if (c["tp"] + c["fn"]) > 0 else 0
        f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0
        print(f"  {lbl:<12} {p:>6.2f} {r:>6.2f} {f1:>6.2f}")
    return acc

In [11]:
# Load base model for comparison
base_model_only = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, dtype=DTYPE, device_map={"": DEVICE}, token=HF_TOKEN
)
base_model_only.eval()

print(f"Evaluating on {len(test_data)} test examples...")
base_acc = evaluate_model(base_model_only, test_data, "BASE MODEL")

del base_model_only
gc.collect()
if DEVICE == "mps":
    torch.mps.empty_cache()

ft_acc = evaluate_model(finetuned_model, test_data, "FINE-TUNED MODEL")

print(f"\nImprovement: {base_acc:.1f}% → {ft_acc:.1f}% (+{ft_acc - base_acc:.1f}pp)")

Loading weights: 100%|██████████| 338/338 [00:00<00:00, 997.59it/s] 
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Evaluating on 18 test examples...

  BASE MODEL
  Label Accuracy: 66.7%
  Label          Prec    Rec     F1
  --------------------------------
  blocking       0.00   0.00   0.00
  suggestion     0.90   0.75   0.82
  warning        0.38   0.75   0.50

  FINE-TUNED MODEL
  Label Accuracy: 72.2%
  Label          Prec    Rec     F1
  --------------------------------
  blocking       0.00   0.00   0.00
  suggestion     0.91   0.83   0.87
  warning        0.43   0.75   0.55

Improvement: 66.7% → 72.2% (+5.6pp)


## Step 10 — Interactive Inference

In [12]:
test_comments = [
    "This SQL query passes user input directly without parameterization — SQL injection risk.",
    "The variable name 'x' is not descriptive. Consider renaming to 'user_count'.",
    "This nested loop runs in O(n²). For large datasets this will be very slow.",
    "Missing docstring on this public method.",
    "No error handling if the database connection fails — the app will crash silently.",
]

print("Fine-tuned model predictions:\n")
for comment in test_comments:
    result = predict(finetuned_model, comment)
    label = result.get("label", "?") if result else "parse error"
    score = result.get("score", "?") if result else "?"
    severity = result.get("severity", "?") if result else "?"
    print(f"  [{label:>10}] score={score} severity={severity}")
    print(f"    '{comment[:75]}'")
    print()

Fine-tuned model predictions:

  [   warning] score=4 severity=high
    'This SQL query passes user input directly without parameterization — SQL in'

  [suggestion] score=2 severity=low
    'The variable name 'x' is not descriptive. Consider renaming to 'user_count''

  [   warning] score=4 severity=high
    'This nested loop runs in O(n²). For large datasets this will be very slow.'

  [suggestion] score=2 severity=low
    'Missing docstring on this public method.'

  [  blocking] score=4 severity=critical
    'No error handling if the database connection fails — the app will crash sil'



## Key Takeaways

**LoRA is efficient.** We trained ~0.14% of parameters and achieved task-specific behaviour. The adapter is a few MB; the base model is shared.

**Synthetic data generation works.** GPT-4o-mini generated ~300 labeled examples in under 30 seconds for ~$0.01. This pattern — use a large model to create training data for a smaller model — is standard in production ML.

**Base vs fine-tuned is the proof.** The base model has no knowledge of our scoring schema. The fine-tuned model consistently outputs valid JSON with the correct label taxonomy.

**What's different from `bert_sentiment`:**
- `bert_sentiment`: encoder model (DistilBERT), discriminative, full fine-tuning
- `lora_finetune`: decoder model (Qwen 1.5B), generative, parameter-efficient (LoRA)

**Production layer:** See `../src/` for the modular implementation and `../cli.py` for the CLI.